# Metric Learning as Cost Deformation

Changing the ground metric changes the quadratic cost matrix and therefore the optimal coupling.  This notebook draws the same empirical measures under three Mahalanobis costs
$$
    c_A(x,y)=(x-y)^\top A(x-y),
$$
from the Euclidean metric to increasingly anisotropic learned metrics.  The gray ellipse shows the unit ball of the metric, so elongated directions are cheaper.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks-figures":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks-figures"))

from figure_style import (
    BLUE,
    RED,
    VIOLET,
    ORANGE,
    GRAY,
    LIGHT_GRAY,
    DIRAC_MARKER_SIZE,
    box_axes,
    draw_point_clouds,
    draw_transport_segments,
    figure_dir,
    interp_color,
    padded_limits,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()

from matplotlib.patches import Ellipse
import ot

NAME = "metric-learning-cost-deformation"
out = figure_dir(NAME)
rng = np.random.default_rng(171)

The point clouds are deliberately close enough that the matching changes because of the metric, not because the geometry is visually disconnected.  POT solves the exact discrete OT problem for each cost.

In [2]:
def rot(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

n = 21
x = np.vstack([
    rng.normal(size=(11, 2)) @ np.diag([0.22, 0.10]) + np.array([-0.72, -0.22]),
    rng.normal(size=(10, 2)) @ np.diag([0.21, 0.11]) + np.array([-0.03, 0.48]),
])
y = np.vstack([
    rng.normal(size=(10, 2)) @ np.diag([0.21, 0.11]) + np.array([0.08, -0.54]),
    rng.normal(size=(11, 2)) @ np.diag([0.22, 0.12]) + np.array([0.80, 0.32]),
])
a = b = np.ones(n) / n

R = rot(np.deg2rad(32))
metrics = {
    "euclidean": np.eye(2),
    "moderate": R @ np.diag([0.32, 2.9]) @ R.T,
    "strong": R @ np.diag([0.075, 8.0]) @ R.T,
}

def cost_matrix(A):
    d = x[:, None, :] - y[None, :, :]
    return np.einsum("...i,ij,...j->...", d, A, d)

plans = {key: ot.emd(a, b, cost_matrix(A)) for key, A in metrics.items()}
print({key: float(P.sum()) for key, P in plans.items()})

{'euclidean': 0.9999999999999999, 'moderate': 0.9999999999999999, 'strong': 0.9999999999999999}


The panels contain no embedded title.  Panel names and the numerical interpretation are placed in the LaTeX caption.

In [3]:
def metric_ellipse(A, scale=0.16):
    w, V = np.linalg.eigh(A)
    order = np.argsort(w)
    w, V = w[order], V[:, order]
    angle = np.degrees(np.arctan2(V[1, 0], V[0, 0]))
    return scale / np.sqrt(w[0]), scale / np.sqrt(w[1]), angle

def draw_metric_plan(P, A, filename):
    fig, ax = plt.subplots(figsize=(2.22, 2.05))
    pairs = [(i, j, float(P[i, j])) for i in range(n) for j in range(n) if P[i, j] > 1e-10]
    draw_transport_segments(ax, x, y, pairs, color=VIOLET, min_width=0.22, max_width=1.00, alpha_scale=0.50, zorder=1)
    draw_point_clouds(ax, x, y, base_size=DIRAC_MARKER_SIZE * 0.86)
    width, height, angle = metric_ellipse(A)
    ell = Ellipse([0.82, -0.71], width, height, angle=angle, fill=False, edgecolor=GRAY, lw=0.85, alpha=0.85)
    ax.add_patch(ell)
    pts = np.vstack([x, y])
    ax.set_xlim(pts[:, 0].min() - 0.30, pts[:, 0].max() + 0.30)
    ax.set_ylim(pts[:, 1].min() - 0.30, pts[:, 1].max() + 0.30)
    ax.set_aspect("equal")
    remove_axes(ax)
    save_pdf(fig, out / filename, pad_inches=0.058)
    plt.close(fig)

for key, A in metrics.items():
    draw_metric_plan(plans[key], A, f"{key}.pdf")
# Backward-compatible alias for older LaTeX/README snippets.
draw_metric_plan(plans["strong"], metrics["strong"], "anisotropic.pdf")